# Análise de dados TCP-CII

### Importação dos parâmetros universais

In [1]:
import importlib.util
from pathlib import Path

path = Path("../../../parametros/config.py").resolve()

spec = importlib.util.spec_from_file_location("parametros", path)
parametros = importlib.util.module_from_spec(spec)
spec.loader.exec_module(parametros)

In [2]:
# Parâmetros importados do arquivo config.py
print(
    "Filtrar por quantidade de alelos TCC1:.........................",
    parametros.filtarar_por_qte_de_alelos_tcc1,
)
print(
    "Parâmetro de filtragem median binding percentile TCC1:.........",
    parametros.parametro_de_filtragem_mbp_tcc1,
)
print(
    "Percentual de match mínimo TCC1:...............................",
    parametros.percent_match_minimo_tcc1,
)

Filtrar por quantidade de alelos TCC1:......................... 10
Parâmetro de filtragem median binding percentile TCC1:......... 5
Percentual de match mínimo TCC1:............................... 95.0


In [3]:
import pandas as pd

In [4]:
df = pd.read_csv('./T CELL/DENV 1 - T Cell Prediction - Class I.csv')
df

,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,VTGKTIHEW,303,311,9,HLA-B*57:01,303,0.01,VTGKTIHEW,VTGKTIHEW,0.997406,0.01
1,1,VTGKTIHEW,303,311,9,HLA-B*58:01,303,0.01,VTGKTIHEW,VTGKTIHEW,0.996622,0.01
2,1,SEMIIPKIY,239,247,9,HLA-B*44:03,239,0.01,SEMIIPKIY,SEMIIPKIY,0.996239,0.01
3,1,SEMIIPKIY,239,247,9,HLA-B*44:02,239,0.01,SEMIIPKIY,SEMIIPKIY,0.992169,0.01
4,1,LSAAIGKAW,42,50,9,HLA-B*57:01,42,0.01,LSAAIGKAW,LSAAIGKAW,0.989554,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
36985,1,PVKEKEENLVKS,337,348,12,HLA-B*53:01,1366,100.00,PVEENLVKS,PVKEKEENLVKS,0.000000,100.00
36986,1,EKEENLVKSMVS,340,351,12,HLA-A*11:01,1369,100.00,ENLVKSMVS,EKEENLVKSMVS,0.000000,100.00
36987,1,EKEENLVKSMVS,340,351,12,HLA-A*24:02,1369,100.00,EKEENKSMV,EKEENLVKSMV,0.000000,100.00
36988,1,EKEENLVKSMVS,340,351,12,HLA-A*32:01,1369,100.00,ENLVKSMVS,EKEENLVKSMVS,0.000000,100.00


## Selecionando Epítopos por median binding percentile.

In [5]:
mbp_minimo = parametros.parametro_de_filtragem_mbp_tcc1

In [6]:
df_mbp_m5 = df[df['median binding percentile'] < mbp_minimo].copy()
print("Filtrando por median binding percentile < ", mbp_minimo)
df_mbp_m5

Filtrando por median binding percentile <  5


,seq #,peptide,start,end,peptide length,allele,peptide index,median binding percentile,netmhcpan_el core,netmhcpan_el icore,netmhcpan_el score,netmhcpan_el percentile
0,1,VTGKTIHEW,303,311,9,HLA-B*57:01,303,0.01,VTGKTIHEW,VTGKTIHEW,0.997406,0.01
1,1,VTGKTIHEW,303,311,9,HLA-B*58:01,303,0.01,VTGKTIHEW,VTGKTIHEW,0.996622,0.01
2,1,SEMIIPKIY,239,247,9,HLA-B*44:03,239,0.01,SEMIIPKIY,SEMIIPKIY,0.996239,0.01
3,1,SEMIIPKIY,239,247,9,HLA-B*44:02,239,0.01,SEMIIPKIY,SEMIIPKIY,0.992169,0.01
4,1,LSAAIGKAW,42,50,9,HLA-B*57:01,42,0.01,LSAAIGKAW,LSAAIGKAW,0.989554,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...
2496,1,LSAAIGKAW,42,50,9,HLA-A*24:02,42,4.90,LSAAIGKAW,LSAAIGKAW,0.003380,4.90
2497,1,TCIWPKSHTLWS,222,233,12,HLA-A*24:02,1251,4.90,TWPKSHTLW,TCIWPKSHTLW,0.003356,4.90
2498,1,SWKSWGKAKII,114,124,11,HLA-A*24:02,801,4.90,SWWGKAKII,SWKSWGKAKII,0.003267,4.90
2499,1,SEKNETWKLARA,204,215,12,HLA-B*44:03,1233,4.90,SEKNETWKA,SEKNETWKLARA,0.003159,4.90


## Agrupando por pepitideos e agregando colunas pertinentes

In [7]:
epitopos_repetidos = (
    df_mbp_m5
    .groupby('peptide', as_index=False)
    .agg(
        start=("start", "first"),
        end=("end", "first"),
        qte_de_alelos=("allele", "nunique"),
        median_binding_percentile=(
            "median binding percentile",
            "median"
        ),
        alelos=(
            "allele",
            lambda x: ", ".join(sorted(x.unique()))
        )
    )
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,AAIGKAWEEGV,44,54,3,2.90,"HLA-A*02:01, HLA-A*02:06, HLA-A*68:02"
1,AAIKDSKAV,186,194,7,2.60,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*6..."
2,AAIKDSKAVH,186,195,2,3.50,"HLA-A*30:02, HLA-B*15:01"
3,ADMGYWIESEK,196,206,2,4.15,"HLA-A*03:01, HLA-A*11:01"
4,ADSPKRLSA,36,44,3,3.20,"HLA-B*08:01, HLA-B*40:01, HLA-B*44:02"
...,...,...,...,...,...,...
597,YTQVCDHRL,175,183,8,2.95,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*2..."
598,YTQVCDHRLM,175,184,4,3.45,"HLA-A*01:01, HLA-B*15:01, HLA-B*57:01, HLA-B*5..."
599,YTQVCDHRLMSA,175,186,1,2.50,HLA-A*01:01
600,YWIESEKNETW,200,210,10,0.80,"HLA-A*23:01, HLA-A*24:02, HLA-A*32:01, HLA-B*3..."


## Filtragem por qte_de_alelos

In [8]:
# Filtro do número de alelos
epitopos_repetidos = epitopos_repetidos[
    epitopos_repetidos["qte_de_alelos"] >= parametros.filtarar_por_qte_de_alelos_tcc1
].reset_index(drop=True)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,ATRLENIMW,60,68,13,3.300,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,AVHADMGYW,193,201,10,1.645,"HLA-A*23:01, HLA-A*26:01, HLA-A*30:02, HLA-A*3..."
2,CIWPKSHTL,223,231,20,1.600,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
3,CTLPPLRFK,316,324,12,1.850,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
4,ECPDNQRAW,142,150,11,1.900,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3..."
5,ESEMIIPKIY,238,247,10,2.550,"HLA-A*01:01, HLA-A*26:01, HLA-A*30:02, HLA-B*3..."
6,ETWKLARASF,208,217,13,2.800,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
7,EVHTWTEQY,24,32,16,1.500,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
8,EVHTWTEQYKF,24,34,10,3.250,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
9,FQADSPKRL,34,42,18,2.450,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."


## Sorting por median_biding_percentile

In [9]:
epitopos_repetidos = (
    epitopos_repetidos
    .sort_values(
        ["median_binding_percentile", "qte_de_alelos"],
        ascending=[True, False]
    )
    .reset_index(drop=True)
)

epitopos_repetidos

,peptide,start,end,qte_de_alelos,median_binding_percentile,alelos
0,QPMEHKYSW,107,115,16,0.545,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
1,YWIESEKNETW,200,210,10,0.800,"HLA-A*23:01, HLA-A*24:02, HLA-A*32:01, HLA-B*3..."
2,IWPKSHTLW,224,232,10,0.960,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*3..."
3,IESEKNETW,202,210,12,1.010,"HLA-A*23:01, HLA-A*24:02, HLA-A*26:01, HLA-A*3..."
4,FVTNEVHTW,20,28,17,1.200,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*2..."
5,ILLENDMKF,78,86,17,1.200,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
6,KAVHADMGY,192,200,11,1.200,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
7,SQHNYRPGY,252,260,14,1.285,"HLA-A*01:01, HLA-A*02:06, HLA-A*03:01, HLA-A*1..."
8,VTGKTIHEW,303,311,14,1.350,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
9,RPQPMEHKYSW,105,115,12,1.350,"HLA-A*23:01, HLA-A*24:02, HLA-A*32:01, HLA-B*0..."


## Separando epítopos e criando arquivo FASTA para IEDB analysis resource

In [10]:
pepitides = epitopos_repetidos.peptide

with open("./peptideos_tcell_1.fasta", "w") as f:
    for i, peptide in enumerate(pepitides, start=1):
        f.write(f">NP {i}\n")
        f.write(f"{peptide}\n")
        
pepitides

0        QPMEHKYSW
1      YWIESEKNETW
2        IWPKSHTLW
3        IESEKNETW
4        FVTNEVHTW
5        ILLENDMKF
6        KAVHADMGY
7        SQHNYRPGY
8        VTGKTIHEW
9      RPQPMEHKYSW
10       RPQPMEHKY
11       KLRDSYTQV
12       HTWTEQYKF
13       EVHTWTEQY
14     KTCIWPKSHTL
15    KIYGGPISQHNY
16       CIWPKSHTL
17      NEVHTWTEQY
18       AVHADMGYW
19       FTTNIWLKL
20     MIRPQPMEHKY
21      QTAGPWHLGK
22       TQTAGPWHL
23       CTLPPLRFK
24       ECPDNQRAW
25      IGADVQNTTF
26      PQPMEHKYSW
27       RSCTLPPLR
28       MIRPQPMEH
29    IRPQPMEHKYSW
30     RSCTLPPLRFK
31       LSAAIGKAW
32       FQADSPKRL
33      TCIWPKSHTL
34      HILLENDMKF
35      TVTGKTIHEW
36       KIYGGPISQ
37       MWKQISNEL
38      ESEMIIPKIY
39       ISNELNHIL
40      IRPQPMEHKY
41      ETWKLARASF
42      KFQADSPKRL
43      FTQTAGPWHL
44     QTAGPWHLGKL
45       SCTLPPLRF
46       SEKNETWKL
47     EVHTWTEQYKF
48       ATRLENIMW
49       TWKLARASF
50       GADVQNTTF
51       VEDYGFGIF
Name: peptid

### Seqkit remove sequências proteicas contendo gaps e *.

In [11]:
# !seqkit grep -s -v -r -p '[-*]' './Fastas/denv1_NS1_proteinas.fa' > DENV1_seq_filter_all.fasta

### Resultado IEDB analysis resource

In [12]:
conservacy_result = pd.read_csv('./ConservancyResult_tcell_1.csv')
conservacy_result

,Epitope #,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,View details
0,1,NP 1,HTWTEQYKF,9,99.80% (1013/1015),88.89%,100.00%,NaN
1,2,NP 2,CIWPKSHTL,9,67.68% (687/1015),77.78%,100.00%,NaN
2,3,NP 3,RPQPMEHKY,9,62.96% (639/1015),77.78%,100.00%,NaN
3,4,NP 4,ISNELNHIL,9,95.76% (972/1015),77.78%,100.00%,NaN
4,5,NP 5,FQADSPKRL,9,99.90% (1014/1015),88.89%,100.00%,NaN
5,6,NP 6,FVTNEVHTW,9,99.61% (1011/1015),88.89%,100.00%,NaN
6,7,NP 7,ILLENDMKF,9,88.47% (898/1015),55.56%,100.00%,NaN
7,8,NP 8,QPMEHKYSW,9,63.65% (646/1015),77.78%,100.00%,NaN
8,9,NP 9,EVHTWTEQY,9,99.80% (1013/1015),88.89%,100.00%,NaN
9,10,NP 10,FTTNIWLKL,9,98.03% (995/1015),88.89%,100.00%,NaN


### Merge da coluna qte_de_alelos ao dataframe conservacy_result

In [13]:
# qte_de_alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "qte_de_alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("View details"))

# alelos
conservacy_result = conservacy_result.merge(
    epitopos_repetidos[["peptide", "alelos"]],
    left_on="Epitope sequence",
    right_on="peptide",
    how="left"
).drop(columns=("peptide")).drop(columns=("Epitope #"))

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos
0,NP 1,HTWTEQYKF,9,99.80% (1013/1015),88.89%,100.00%,21,"HLA-A*01:01, HLA-A*02:06, HLA-A*11:01, HLA-A*2..."
1,NP 2,CIWPKSHTL,9,67.68% (687/1015),77.78%,100.00%,20,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
2,NP 3,RPQPMEHKY,9,62.96% (639/1015),77.78%,100.00%,19,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
3,NP 4,ISNELNHIL,9,95.76% (972/1015),77.78%,100.00%,19,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
4,NP 5,FQADSPKRL,9,99.90% (1014/1015),88.89%,100.00%,18,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2..."
5,NP 6,FVTNEVHTW,9,99.61% (1011/1015),88.89%,100.00%,17,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*2..."
6,NP 7,ILLENDMKF,9,88.47% (898/1015),55.56%,100.00%,17,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."
7,NP 8,QPMEHKYSW,9,63.65% (646/1015),77.78%,100.00%,16,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2..."
8,NP 9,EVHTWTEQY,9,99.80% (1013/1015),88.89%,100.00%,16,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2..."
9,NP 10,FTTNIWLKL,9,98.03% (995/1015),88.89%,100.00%,16,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0..."


### Gerando a coluna percent_match para filtrar os epitopos com percentagem de match maior que 50%

In [14]:
col = "Percent of protein sequence matches at identity <= 100%"

conservacy_result["percent_match"] = (
    conservacy_result[col]
    .astype(str)
    .str.extract(r"(\d+(?:\.\d+)?)")[0]
    .astype(float)
)

conservacy_result

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 1,HTWTEQYKF,9,99.80% (1013/1015),88.89%,100.00%,21,"HLA-A*01:01, HLA-A*02:06, HLA-A*11:01, HLA-A*2...",99.80
1,NP 2,CIWPKSHTL,9,67.68% (687/1015),77.78%,100.00%,20,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2...",67.68
2,NP 3,RPQPMEHKY,9,62.96% (639/1015),77.78%,100.00%,19,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2...",62.96
3,NP 4,ISNELNHIL,9,95.76% (972/1015),77.78%,100.00%,19,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0...",95.76
4,NP 5,FQADSPKRL,9,99.90% (1014/1015),88.89%,100.00%,18,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2...",99.90
5,NP 6,FVTNEVHTW,9,99.61% (1011/1015),88.89%,100.00%,17,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*2...",99.61
6,NP 7,ILLENDMKF,9,88.47% (898/1015),55.56%,100.00%,17,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0...",88.47
7,NP 8,QPMEHKYSW,9,63.65% (646/1015),77.78%,100.00%,16,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2...",63.65
8,NP 9,EVHTWTEQY,9,99.80% (1013/1015),88.89%,100.00%,16,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2...",99.80
9,NP 10,FTTNIWLKL,9,98.03% (995/1015),88.89%,100.00%,16,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:03, HLA-A*0...",98.03


### Sort e filtragem por percent_match e presença em alelos

In [15]:
# Filtro do Percent match
conservacy_result_filtered = (
    conservacy_result[conservacy_result["percent_match"] >= parametros.percent_match_minimo_tcc1]
    .sort_values(
            by="percent_match", 
            ascending=False
        )
    ).reset_index(drop=True)

conservacy_result_filtered

,Epitope name,Epitope sequence,Epitope length,Percent of protein sequence matches at identity <= 100%,Minimum identity,Maximum identity,qte_de_alelos,alelos,percent_match
0,NP 5,FQADSPKRL,9,99.90% (1014/1015),88.89%,100.00%,18,"HLA-A*02:01, HLA-A*02:03, HLA-A*02:06, HLA-A*2...",99.90
1,NP 1,HTWTEQYKF,9,99.80% (1013/1015),88.89%,100.00%,21,"HLA-A*01:01, HLA-A*02:06, HLA-A*11:01, HLA-A*2...",99.80
2,NP 9,EVHTWTEQY,9,99.80% (1013/1015),88.89%,100.00%,16,"HLA-A*01:01, HLA-A*03:01, HLA-A*11:01, HLA-A*2...",99.80
3,NP 44,RSCTLPPLR,9,99.80% (1013/1015),88.89%,100.00%,10,"HLA-A*03:01, HLA-A*11:01, HLA-A*30:01, HLA-A*3...",99.80
4,NP 49,KFQADSPKRL,10,99.80% (1013/1015),90.00%,100.00%,10,"HLA-A*02:03, HLA-A*02:06, HLA-A*23:01, HLA-A*2...",99.80
5,NP 40,NEVHTWTEQY,10,99.80% (1013/1015),90.00%,100.00%,10,"HLA-A*01:01, HLA-A*26:01, HLA-A*30:02, HLA-A*6...",99.80
6,NP 51,EVHTWTEQYKF,11,99.70% (1012/1015),90.91%,100.00%,10,"HLA-A*01:01, HLA-A*23:01, HLA-A*24:02, HLA-A*2...",99.70
7,NP 6,FVTNEVHTW,9,99.61% (1011/1015),88.89%,100.00%,17,"HLA-A*01:01, HLA-A*02:01, HLA-A*02:06, HLA-A*2...",99.61
8,NP 41,AVHADMGYW,9,99.61% (1011/1015),88.89%,100.00%,10,"HLA-A*23:01, HLA-A*26:01, HLA-A*30:02, HLA-A*3...",99.61
9,NP 14,SQHNYRPGY,9,98.82% (1003/1015),88.89%,100.00%,14,"HLA-A*01:01, HLA-A*02:06, HLA-A*03:01, HLA-A*1...",98.82
